## Data (not in git)
- Podcast: NPR Planet Money, Episode 1,000 (2020)
- Transcript: https://www.npr.org/transcripts/857106873
- Put the files in `samples/` (this folder is ignored by git):
  - `samples/planet_money_1000.mp3`: the episode audio from the podcast feed
  - `samples/transcript_planet_money_1000.txt`: the transcript text, from "BYLINE: This is PLANET MONEY from NPR." to the last line
- The podcast feed version has ads that are not in the transcript (one at the start, three later). The notebook skips the first 25 seconds and removes the other ads before WER.

In [5]:
from faster_whisper import WhisperModel
from pathlib import Path

from faster_whisper import decode_audio
from IPython.display import Audio

In [4]:
model = WhisperModel("small", device="cpu", compute_type="int8", cpu_threads=8)

Help on WhisperModel in module faster_whisper.transcribe object:

class WhisperModel(builtins.object)
 |  WhisperModel(model_size_or_path: str, device: str = 'auto', device_index: Union[int, List[int]] = 0, compute_type: str = 'default', cpu_threads: int = 0, num_workers: int = 1, download_root: Optional[str] = None, local_files_only: bool = False, files: dict = None, revision: Optional[str] = None, use_auth_token: Union[str, bool, NoneType] = None, **model_kwargs)
 |
 |  Methods defined here:
 |
 |  __init__(self, model_size_or_path: str, device: str = 'auto', device_index: Union[int, List[int]] = 0, compute_type: str = 'default', cpu_threads: int = 0, num_workers: int = 1, download_root: Optional[str] = None, local_files_only: bool = False, files: dict = None, revision: Optional[str] = None, use_auth_token: Union[str, bool, NoneType] = None, **model_kwargs)
 |      Initializes the Whisper model.
 |
 |      Args:
 |        model_size_or_path: Size of the model to use (tiny, tiny.en, b

In [ ]:
SAMPLES = Path("../samples")    
SAMPLE_RATE = 16000              # Whisper expects 16 kHz !!

In [ ]:
audio = decode_audio(str(SAMPLES / "planet_money_1000.mp3")) # https://www.npr.org/transcripts/857106873
print(f"{len(audio) / SAMPLE_RATE / 60:.1f} minutes")


27.7 minutes


In [ ]:
Audio(audio[: 30 * SAMPLE_RATE], rate=SAMPLE_RATE)    # listen to the first 30 seconds

In [10]:
transcript = (SAMPLES / "transcript_planet_money_1000.txt").read_text(encoding="utf-8")

print(len(transcript.split()), "words")
print(transcript[:500])        # the first 500 characters

4913 words
UNIDENTIFIED PERSON #1, BYLINE: This is PLANET MONEY from NPR.

(SOUNDBITE OF SONG, "PLANET MONEY (IT'S THEIR 1,000TH EPISODE)")

SCOTT LANE AND DEAU EYES: (Singing) PLANET MONEY - it's their 1,000th episode.

KENNY MALONE, HOST:

Yeah, it's our 1,000th episode. And to be honest, we spent a lot of time thinking about ways to avoid doing this episode.

SARAH GONZALEZ, HOST:

There's a lot going on right now. So at one point, we were like, what if we just skip episode 1,000, go from Episode 999 to


In [12]:
help(model)

Help on WhisperModel in module faster_whisper.transcribe object:

class WhisperModel(builtins.object)
 |  WhisperModel(model_size_or_path: str, device: str = 'auto', device_index: Union[int, List[int]] = 0, compute_type: str = 'default', cpu_threads: int = 0, num_workers: int = 1, download_root: Optional[str] = None, local_files_only: bool = False, files: dict = None, revision: Optional[str] = None, use_auth_token: Union[str, bool, NoneType] = None, **model_kwargs)
 |
 |  Methods defined here:
 |
 |  __init__(self, model_size_or_path: str, device: str = 'auto', device_index: Union[int, List[int]] = 0, compute_type: str = 'default', cpu_threads: int = 0, num_workers: int = 1, download_root: Optional[str] = None, local_files_only: bool = False, files: dict = None, revision: Optional[str] = None, use_auth_token: Union[str, bool, NoneType] = None, **model_kwargs)
 |      Initializes the Whisper model.
 |
 |      Args:
 |        model_size_or_path: Size of the model to use (tiny, tiny.en, b

In [17]:
clip = audio[: 5 * 60 * SAMPLE_RATE]
text_1=model.transcribe(clip,
                                                                     task="transcribe")

type(text_1)
text_1

(<generator object WhisperModel.generate_segments at 0x000001C74FE71A40>,
 TranscriptionInfo(language='en', language_probability=0.993934690952301, duration=300.0, duration_after_vad=300.0, all_language_probs=[('en', 0.993934690952301), ('es', 0.0012807565508410335), ('ru', 0.000798661494627595), ('de', 0.0006158689502626657), ('fr', 0.0005141333676874638), ('la', 0.0004075834876857698), ('ko', 0.00032057991484180093), ('pt', 0.00026195330428890884), ('ar', 0.00022832986724097282), ('zh', 0.00018904567696154118), ('ja', 0.00017596107500139624), ('nn', 0.00014806247781962156), ('id', 0.00012953291297890246), ('cy', 8.338441693922505e-05), ('it', 8.18103872006759e-05), ('vi', 7.491036376450211e-05), ('haw', 6.931439565960318e-05), ('nl', 6.732529436703771e-05), ('th', 5.082290226710029e-05), ('km', 5.018425508751534e-05), ('sn', 4.621731204679236e-05), ('mi', 4.080523285665549e-05), ('el', 4.014886144432239e-05), ('pl', 3.4488468372728676e-05), ('jw', 3.171384014422074e-05), ('tr', 3.133

In [19]:
import time

START = 25                                            # skip the ad at the start (seconds)
clip = audio[START * SAMPLE_RATE : (START + 5 * 60) * SAMPLE_RATE]

t0 = time.perf_counter()
segments, info = model.transcribe(clip, language="en", task="transcribe",
                                  beam_size=5, vad_filter=True)

texts = []
for s in segments:                             # transcription happens while looping
    print(f"[{s.start:6.1f} -> {s.end:6.1f}] {s.text.strip()}")
    texts.append(s.text.strip())

elapsed = time.perf_counter() - t0
duration = len(clip) / SAMPLE_RATE
print(f"\naudio {duration:.0f}s  took {elapsed:.0f}s  RTF {elapsed / duration:.2f}")

hypothesis = " ".join(texts)

[   0.6 ->   16.0] This is Planet Money from NPR.
[  16.0 ->   17.9] Yeah, it's our 1,000th episode.
[  17.9 ->   20.9] And to be honest, we spent a lot of time thinking
[  20.9 ->   23.7] about ways to avoid doing this episode.
[  23.7 ->   25.0] There's a lot going on right now.
[  25.0 ->   26.5] So at one point we were like,
[  26.5 ->   28.8] what if we just skip episode 1,000,
[  28.8 ->   31.8] go from episode 999 to episode 1,001,
[  31.8 ->   33.9] and just do the big 1,000th later
[  33.9 ->   35.7] when we can throw a real party?
[  35.7 ->   36.8] Or my personal suggestion,
[  36.8 ->   39.4] we could suddenly switch to decimal system,
[  39.4 ->   44.4] you know, like, suddenly it's 999.5, 0.75, 0.875.
[  45.1 ->   46.5] We don't ever have to get to 1,000.
[  46.5 ->   48.1] We can stay young forever.
[  48.1 ->   49.0] But then we were like, all right,
[  49.0 ->   50.6] let's see what the listeners have to say.
[  50.6 ->   53.2] And this is when we asked you to send us


In [20]:
import re

import jiwer
from transformers import WhisperTokenizer

reference = transcript
reference = re.sub(r"^\(.*\)\s*$", "", reference, flags=re.M)             # lines that are only a sound note, e.g. (SOUNDBITE OF MUSIC)
reference = re.sub(r"^[A-Z][A-Z0-9 .,'#&-]*:", "", reference, flags=re.M)  # speaker names, e.g. KENNY MALONE, HOST:

tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-large-v3-turbo")
ref_norm = tokenizer.normalize(reference)

print(len(ref_norm.split()), "words")      # should be 4435
print(ref_norm[:300])

4435 words
this is planet money from npr planet money it is their 1000th episode yeah it is our 1000th episode and to be honest we spent a lot of time thinking about ways to avoid doing this episode there is a lot going on right now so at one point we were like what if we just skip episode 1000 go from episode


In [21]:
import time

START = 25                                    # skip the ad at the start (seconds)
episode = audio[START * SAMPLE_RATE :]

t0 = time.perf_counter()
segments, info = model.transcribe(episode, language="en", task="transcribe",
                                  beam_size=5, vad_filter=True)
texts = []
for s in segments:
    print(f"[{s.start + START:7.1f}] {s.text.strip()}")
    texts.append(s.text.strip())
elapsed = time.perf_counter() - t0

hypothesis = " ".join(texts)
print(f"\naudio {info.duration:.0f}s  took {elapsed:.0f}s  RTF {elapsed / info.duration:.2f}")

[   25.6] This is Planet Money from NPR.
[   41.0] Yeah, it's our 1,000th episode.
[   42.9] And to be honest, we spent a lot of time thinking
[   45.9] about ways to avoid doing this episode.
[   48.7] There's a lot going on right now.
[   50.0] So at one point we were like,
[   51.5] what if we just skip episode 1,000,
[   53.8] go from episode 999 to episode 1,001,
[   56.8] and just do the big 1,000th later
[   58.9] when we can throw a real party?
[   60.7] Or my personal suggestion,
[   61.8] we could suddenly switch to decimal system,
[   65.3] suddenly it's 999.5, 0.75, 0.875.
[   70.1] We don't ever have to get to 1,000.
[   71.6] We can stay young forever.
[   73.1] But then we were like, all right,
[   74.0] let's see what the listeners have to say.
[   75.6] And this is when we asked you to send us
[   78.2] your favorite moments from Planet Money.
[   80.1] And we listened to a bunch of your suggestions.
[   81.6] And as we were going through them,
[   82.7] we noticed we'

In [22]:
hyp_norm = tokenizer.normalize(hypothesis)

out = jiwer.process_words(ref_norm, hyp_norm)
print(f"WER {100 * out.wer:.1f} %")
print(f"wrong {out.substitutions}, missing {out.deletions}, extra {out.insertions} "
      f"(reference: {len(ref_norm.split())} words)")

WER 16.1 %
wrong 133, missing 115, extra 466 (reference: 4435 words)


In [23]:
for chunk in out.alignments[0]:
    if chunk.type == "insert":
        n = chunk.hyp_end_idx - chunk.hyp_start_idx
        if n >= 10:
            words = out.hypotheses[0][chunk.hyp_start_idx : chunk.hyp_end_idx]
            print(f"{n} extra words: {' '.join(words[:40])} ...\n")

207 extra words: election season feels messier than ever attack ads misinformation ai npr is your antidote our network of real human journalists keeps you focused on the facts and with npr plus you directly support journalists on the ground asking questions in ...

143 extra words: american journalist austin tice vanished in syria over a decade ago for years we have been searching for him and now we finally have some answers this is the prison that austin was in 0 my god we found austin ...

42 extra words: on the next through line from npr i was surprised that this man every single day takes a chair and goes to the troubles and debris of his house and he starts to tell stories to his children to his ...



In [24]:
# the three long "extra" blocks printed above are ads, not part of the episode
ads = [c for c in out.alignments[0]
       if c.type == "insert" and c.hyp_end_idx - c.hyp_start_idx >= 40]

hyp_words = out.hypotheses[0]
kept = [w for i, w in enumerate(hyp_words)
        if not any(c.hyp_start_idx <= i < c.hyp_end_idx for c in ads)]

out_clean = jiwer.process_words(ref_norm, " ".join(kept))
print(f"removed {len(hyp_words) - len(kept)} ad words")
print(f"WER {100 * out_clean.wer:.1f} %")
print(f"wrong {out_clean.substitutions}, missing {out_clean.deletions}, extra {out_clean.insertions}")

removed 392 ad words
WER 7.3 %
wrong 133, missing 115, extra 74
